In [ ]:
import os
import time
import warnings

import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hyperparameter tuning -- Random Forest, XGBoost, SVC

Nested-CV hyperparameter tuning for the three feature-based ML models. The
other models are tuned with the same validation protocol, objective and
trial budget in `07b_tuning_features_dl.ipynb` (GANDALF, CategoryEmbedding),
`07c_tuning_features_tabpfn.ipynb` (TabPFN) and
`07d_tuning_timeseries_dl.ipynb` (HydraMultiROCKET, LSTM, TSSequencer); the
notebooks differ only in the per-fold trainer and search space. Each
model's tuning cells can be run independently; outputs are merged into the
shared CSVs (see "Save outputs").

## Validation protocol

Uses the 5 stratified-group CV folds from `indices_cv_fold_lookup.csv`
(the same folds as `05a_feature_classification_ml.ipynb`):

- **Outer loop**: each fold in turn is the test fold (5 outer folds).
- **Inner loop**: for test fold `folds[i]`, the next `N_INNER_FOLDS` (= 2)
  folds, `folds[(i+1) % 5]` and `folds[(i+2) % 5]`, are inner validation
  folds. For each, the model is trained on the 3 remaining non-test folds
  and scored with `f1_macro` on that validation fold; an Optuna trial's
  score is the mean over the 2 inner folds. The test fold is not used
  during the search.
- The best hyperparameters are retrained on all 4 non-test folds and
  evaluated once on the test fold.

## Search space

Class imbalance handling is fixed, not tuned: `class_weight="balanced"`
for RF/SVC, and `sample_weight=compute_sample_weight("balanced", ...)` at
fit time for XGBoost.

Every model (here and in `07b`-`07d`, apart from TabPFN in `07c`) tunes two
parameters: one that sets capacity (the size or flexibility of the fitted
function) and one that regularises the fit through a separate mechanism
(a penalty, shrinkage, randomisation or dropout).

- **Random Forest**: `min_samples_leaf` (capacity, integer, 1-10, log
  scale) + `max_features` (regularisation by randomisation, integer, 1-60,
  log scale). `n_estimators=100` and `max_depth=None` are fixed at their
  defaults (`RF_FIXED_PARAMS`). The upper bound of `min_samples_leaf` is
  low because the rarest classes have only a few windows per training
  split (3-4 Running windows at Burst 1).
- **XGBoost**: `max_depth` (capacity, integer, 3-20) + `learning_rate`
  (regularisation by shrinkage, 0.01-1, log scale). `n_estimators=100` is
  fixed at its default (`XGB_FIXED_PARAMS`).
- **SVC** (RBF kernel): `gamma` (capacity, 1e-4-1, log scale) + `C`
  (regularisation, 0.1-100, log scale). Features are standardised with a
  `StandardScaler` fitted on each training split only, so the `gamma`
  range is on the standardised scale.

## Default-hyperparameter control

Each model's default configuration (`RF_DEFAULT_PARAMS`,
`XGB_DEFAULT_PARAMS`, `SVC_DEFAULT_PARAMS`, matching
`05a_feature_classification_ml.ipynb`) is evaluated on the same folds, with
the same code path and seed as the tuned model, so tuned vs default is a
like-for-like comparison (both use the same `RF_FIXED_PARAMS` /
`XGB_FIXED_PARAMS`):

- **Validation side**: `default_val_f1_macro`, the default's mean inner
  `f1_macro`, directly comparable to the tuned `val_f1_macro`. For RF
  (`min_samples_leaf=1`, `max_features=7`, i.e. sklearn's `"sqrt"` for
  the 60 features) and XGBoost (`max_depth=6`, `learning_rate=0.3`) the
  default lies inside the search space and is enqueued as the first
  Optuna trial. For SVC (`gamma="scale"`) the default cannot be expressed
  in the search space, so it is evaluated directly via
  `eval_svc_cv_folds`.
- **Test side**: the default is also retrained on the 4 non-test folds and
  evaluated on the test fold (`retrain_and_evaluate_rf` / `_xgb` / `_svc`,
  shared with the tuned retrain). These rows (`model="<model>_default"`)
  are saved to separate `..._tuned_default_control_*.csv` files, so the
  `..._tuned_metrics.csv` / `..._tuned_confusion.csv` files read by
  `12b_publication_plots_fig3.R` contain only tuned rows.

## Objective and budget

The objective is `f1_macro`, so rare behaviour classes count equally with
common ones. Optuna uses a seeded multivariate TPE sampler and the same
`N_TRIALS` (= 50) for every model.

## Scope

Burst_1/uncorrected only, behaviour (multiclass) target only.


## Load the main CV fold lookup and set tuning constants

In [ ]:
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
DATASETS = [
    ("burst_1", "uncorrected"),
]
N_TRIALS = 50
RANDOM_STATE = 42
# Number of inner validation folds whose f1_macro is averaged per Optuna
# trial (see the notebook intro for the fold pairing).
N_INNER_FOLDS = 2
# Number of feature columns in the tuned feature files. RF's default
# max_features="sqrt" is int(sqrt(60)) = 7, so RF_DEFAULT_PARAMS below
# reproduces it only for 60 features (checked in process_parquet_file_rf).
N_FEATURES = 60
# RF: parameters fixed at their defaults in every trial and retrain, as in
# 05a_feature_classification_ml.ipynb (n_estimators=100, sklearn's default
# max_depth=None).
RF_FIXED_PARAMS = {"n_estimators": 100, "max_depth": None}
# Default values of the two tuned RF parameters (sklearn's
# min_samples_leaf=1 and max_features="sqrt" = 7 for 60 features). Both
# lie inside the search space, so the default is enqueued as one of the
# N_TRIALS trials (see tune_and_evaluate_rf_fold). Only the tuned keys are
# listed, so the enqueued trial can be found by matching trial.params.
RF_DEFAULT_PARAMS = {"min_samples_leaf": 1, "max_features": 7}
# XGBoost: n_estimators fixed at XGBoost's default (100) in every trial and
# retrain.
XGB_FIXED_PARAMS = {"n_estimators": 100}
# Default values of the two tuned XGBoost parameters (XGBoost's own
# defaults, as in 05a_feature_classification_ml.ipynb). Both lie inside the
# search space, so the default is enqueued as one of the N_TRIALS trials
# (see tune_and_evaluate_xgb_fold).
XGB_DEFAULT_PARAMS = {"max_depth": 6, "learning_rate": 0.3}
# Default SVC configuration, as in 05a_feature_classification_ml.ipynb
# (sklearn defaults C=1.0, gamma="scale"). gamma="scale" is data-dependent
# (1 / (n_features * X.var())) and cannot be expressed in the tuned
# suggest_float("gamma", ...) range, so it is evaluated directly in
# tune_and_evaluate_svc_fold rather than enqueued as a trial.
SVC_DEFAULT_PARAMS = {"C": 1.0, "gamma": "scale"}

fold_lookup_all = pd.read_csv(FOLD_LOOKUP_PATH)
print(
    f"Loaded fold lookup: {len(fold_lookup_all)} rows, "
    f"bursts={sorted(fold_lookup_all['burst'].unique())}"
)

## RF Optuna search + final evaluation, for one outer test fold

In [ ]:
def eval_rf_cv_folds(
    cv_folds, min_samples_leaf, max_features, use_class_weight=True
):
    """Fit+score one RF hyperparameter setting across all of cv_folds (the
    N_INNER_FOLDS inner train/val splits), returning the mean f1_macro.

    Used by make_rf_objective's Optuna trials, including the enqueued
    RF_DEFAULT_PARAMS trial. RF_FIXED_PARAMS (n_estimators, max_depth) are
    the same in every trial.
    """
    cw = "balanced" if use_class_weight else None
    scores = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for X_train, y_train, X_val, y_val in cv_folds:
            model = RandomForestClassifier(
                **RF_FIXED_PARAMS,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                class_weight=cw,
                random_state=RANDOM_STATE,
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))
    return float(np.mean(scores))


def make_rf_objective(cv_folds, use_class_weight=True):
    """Build an Optuna objective that returns the mean f1_macro across
    the N_INNER_FOLDS inner (train, val) splits. Searches
    min_samples_leaf (capacity, integer, 1-10, log scale) and max_features
    (regularisation, integer, 1-60, log scale). RF_DEFAULT_PARAMS lies
    inside this space and is enqueued as a trial (see
    tune_and_evaluate_rf_fold).

    Args:
        cv_folds (list): list of (X_train, y_train, X_val, y_val) tuples,
            one per inner fold.
    """

    def objective(trial):
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10, log=True)
        max_features = trial.suggest_int("max_features", 1, N_FEATURES, log=True)
        return eval_rf_cv_folds(
            cv_folds,
            min_samples_leaf,
            max_features,
            use_class_weight=use_class_weight,
        )

    return objective


def retrain_and_evaluate_rf(
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    params,
    class_names,
    use_class_weight=True,
):
    """Fit one RF hyperparameter setting on X_train_val/y_train_val and
    evaluate once on X_test/y_test, returning (metrics_row, confusion_rows)
    -- confusion_rows is a list of dicts without a "model" key (the caller
    tags that).

    Shared by tune_and_evaluate_rf_fold's two final retrains (tuned winner
    and RF_DEFAULT_PARAMS control), so both use the same code path and
    differ only in the hyperparameters.
    """
    cw = "balanced" if use_class_weight else None
    n_classes = len(class_names)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model = RandomForestClassifier(
            **RF_FIXED_PARAMS,
            **params,
            class_weight=cw,
            random_state=RANDOM_STATE,
        )
        model.fit(X_train_val, y_train_val)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        try:
            roc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        metrics_row = {
            "min_samples_leaf": params["min_samples_leaf"],
            "max_features": params["max_features"],
            "roc_auc": roc,
            "accuracy": model.score(X_test, y_test),
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "precision": precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "precision_macro": precision_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "time": time.time() - t0,
        }

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
        confusion_rows = [
            {
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(n_classes)
            for j in range(n_classes)
        ]

    return metrics_row, confusion_rows


def tune_and_evaluate_rf_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for RF: Optuna-search hyperparameters
    scored by the mean f1_macro across N_INNER_FOLDS validation folds (see
    make_rf_objective), then retrain the winner on all non-test folds
    combined and evaluate once on the test fold. Final metrics and
    confusion matrix follow features_ml_models in
    05a_feature_classification_ml.ipynb.

    One Optuna study (seeded multivariate TPE) is created per outer test
    fold, named by burst/correction_type/test_fold, and returned for the
    post-hoc "Inspect best study per dataset" cell.

    RF_DEFAULT_PARAMS is used as a control in two ways:

    1. Enqueued as the first Optuna trial (study.enqueue_trial), so its
       score on the same cv_folds is `default_val_f1_macro` (comparable
       to `val_f1_macro`).
    2. Retrained on the same X_train_val/y_train_val and evaluated on the
       same X_test/y_test as the tuned winner (via retrain_and_evaluate_rf).

    Args:
        cv_folds (list): list of (X_train, y_train, X_val, y_val) tuples
            for the inner validation search (see make_rf_objective).
        X_train_val, y_train_val: all non-test-fold data combined, for the
            final retrain with the winning hyperparameters.
        X_test, y_test: the held-out test fold, touched only once, after
            hyperparameters are already fixed.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). metrics_df/confusion_df cover
            the tuned winner (model="RF"); default_metrics_df/
            default_confusion_df cover the RF_DEFAULT_PARAMS retrain
            (model="RF_default"), kept separate so they are saved to the
            ..._tuned_default_control_*.csv files (see the save cell).
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )
    study.enqueue_trial(RF_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_rf_objective(cv_folds, use_class_weight=use_class_weight),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    # Validation score of the enqueued default trial, found by matching
    # params.
    default_trial = next(
        (t for t in study.trials if t.params == RF_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    tuned_row, tuned_conf_rows = retrain_and_evaluate_rf(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        best_params,
        class_names,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "RF",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "RF", **row} for row in tuned_conf_rows]

    # Default-hyperparameter control, retrained on the same split as the
    # tuned retrain above.
    default_row, default_conf_rows = retrain_and_evaluate_rf(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        RF_DEFAULT_PARAMS,
        class_names,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "RF_default"
    default_confusion_data = [
        {"model": "RF_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## RF tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_rf(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for Random Forest
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Follows process_parquet_file in 05a_feature_classification_ml.ipynb, but
    each of the 5 outer test folds gets its own Optuna search (see
    make_rf_objective) before the final retrain-and-test step.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup_all.loc[
        fold_lookup_all["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "behaviour", "fold"],
    ]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(["Ind_ID", "new_burst", "behaviour", "fold"], axis=1)
    # RF_DEFAULT_PARAMS (max_features=7) equals sklearn's "sqrt" only for
    # N_FEATURES columns.
    assert X_full.shape[1] == N_FEATURES, (
        f"Expected {N_FEATURES} feature columns, found {X_full.shape[1]}"
    )
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        # The N_INNER_FOLDS folds following test_fold are the inner
        # validation folds; each inner round trains on the remaining 3
        # folds (excluding test_fold and that round's validation fold).
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_rf_fold(
            cv_folds,
            X_full[non_test],
            y_enc[non_test],
            X_full[te],
            y_enc[te],
            class_names,
            burst,
            correction_type,
            test_fold,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        # Kept for the post-hoc "Inspect best study per dataset" cell.
        out["studies"][test_fold] = study

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "min_samples_leaf",
        "max_features",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    # Default-control metrics: no val_f1_macro/default_val_f1_macro/
    # search_time columns, since no search is run for them.
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "min_samples_leaf",
        "max_features",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }


## Run tuning across datasets

In [ ]:
all_results_rf = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_rf(pq_file_path, fold_lookup_all, n_trials=N_TRIALS)
    all_results_rf.append(result)

multiclass_metrics_rf = pd.concat(
    [r["multiclass_metrics"] for r in all_results_rf], ignore_index=True
)
multiclass_conf_rf = pd.concat(
    [r["multiclass_conf"] for r in all_results_rf], ignore_index=True
)

# Default-hyperparameter control (model="RF_default"), retrained and
# evaluated on the same splits as the tuned winner. Kept separate from
# multiclass_metrics_rf/multiclass_conf_rf (saved to separate files).
default_metrics_rf = pd.concat(
    [r["default_metrics"] for r in all_results_rf], ignore_index=True
)
default_conf_rf = pd.concat(
    [r["default_conf"] for r in all_results_rf], ignore_index=True
)

# Kept in memory (not saved) for the "Inspect best study per dataset"
# cell: all_studies_rf[dataset_key] is a dict of {test_fold: study}.
all_studies_rf = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_rf)
}

## XGBoost Optuna search + final evaluation, for one outer test fold

In [ ]:
def eval_xgb_cv_folds(cv_folds, max_depth, learning_rate, use_class_weight=True):
    """Fit+score one XGBoost hyperparameter setting across all of cv_folds
    (the N_INNER_FOLDS inner train/val splits), returning the mean
    f1_macro.

    Mirrors eval_rf_cv_folds; used by make_xgb_objective's Optuna trials
    (including the enqueued XGB_DEFAULT_PARAMS trial). XGB_FIXED_PARAMS
    (n_estimators) is the same in every trial. XGBoost has no
    class_weight argument, so class imbalance is handled via balanced
    sample_weight at fit time, as in 05a_feature_classification_ml.ipynb.
    """
    scores = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for X_train, y_train, X_val, y_val in cv_folds:
            model = XGBClassifier(
                **XGB_FIXED_PARAMS,
                max_depth=max_depth,
                learning_rate=learning_rate,
                random_state=RANDOM_STATE,
            )
            if use_class_weight:
                model.fit(
                    X_train,
                    y_train,
                    sample_weight=compute_sample_weight("balanced", y_train),
                )
            else:
                model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))
    return float(np.mean(scores))


def make_xgb_objective(cv_folds, use_class_weight=True):
    """Build an Optuna objective for XGBoost that returns the mean
    f1_macro across the N_INNER_FOLDS inner validation folds. Searches
    max_depth (capacity, integer, 3-20) and learning_rate (regularisation
    by shrinkage, 0.01-1, log scale). XGB_DEFAULT_PARAMS lies inside this
    space and is enqueued as a trial (see tune_and_evaluate_xgb_fold).
    """

    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 3, 20)
        learning_rate = trial.suggest_float("learning_rate", 0.01, 1, log=True)
        return eval_xgb_cv_folds(
            cv_folds,
            max_depth,
            learning_rate,
            use_class_weight=use_class_weight,
        )

    return objective


def retrain_and_evaluate_xgb(
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    params,
    class_names,
    use_class_weight=True,
):
    """Fit one XGBoost hyperparameter setting on X_train_val/y_train_val
    and evaluate once on X_test/y_test, returning (metrics_row,
    confusion_rows) -- confusion_rows is a list of dicts without a
    "model" key (the caller tags that).

    Mirrors retrain_and_evaluate_rf. Shared by tune_and_evaluate_xgb_fold's
    two final retrains: the tuned winner and the XGB_DEFAULT_PARAMS control.
    """
    n_classes = len(class_names)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model = XGBClassifier(
            **XGB_FIXED_PARAMS, **params, random_state=RANDOM_STATE
        )
        if use_class_weight:
            model.fit(
                X_train_val,
                y_train_val,
                sample_weight=compute_sample_weight("balanced", y_train_val),
            )
        else:
            model.fit(X_train_val, y_train_val)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        try:
            roc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        metrics_row = {
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "roc_auc": roc,
            "accuracy": model.score(X_test, y_test),
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "precision": precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "precision_macro": precision_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "time": time.time() - t0,
        }

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
        confusion_rows = [
            {
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(n_classes)
            for j in range(n_classes)
        ]

    return metrics_row, confusion_rows


def tune_and_evaluate_xgb_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for XGBoost. Same protocol as
    tune_and_evaluate_rf_fold: XGB_DEFAULT_PARAMS is enqueued as the first
    Optuna trial (study.enqueue_trial) and that trial's value is used as
    default_val_f1_macro. XGB_DEFAULT_PARAMS is also retrained on the same
    split as the tuned winner (see retrain_and_evaluate_xgb).

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). Same shape as
            tune_and_evaluate_rf_fold's return, with model="XGB" /
            "XGB_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )
    study.enqueue_trial(XGB_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_xgb_objective(cv_folds, use_class_weight=use_class_weight),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    # Validation score of the enqueued default trial, found by matching
    # params.
    default_trial = next(
        (t for t in study.trials if t.params == XGB_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    tuned_row, tuned_conf_rows = retrain_and_evaluate_xgb(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        best_params,
        class_names,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "XGB",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "XGB", **row} for row in tuned_conf_rows]

    # Default-hyperparameter control, retrained on the same split as the
    # tuned retrain above.
    default_row, default_conf_rows = retrain_and_evaluate_xgb(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        XGB_DEFAULT_PARAMS,
        class_names,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "XGB_default"
    default_confusion_data = [
        {"model": "XGB_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## XGBoost tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_xgb(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for XGBoost
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Same structure as process_parquet_file_rf, with
    tune_and_evaluate_xgb_fold as the per-fold trainer; the hyperparameter
    columns (n_estimators, max_depth) are the same as RF's.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup_all.loc[
        fold_lookup_all["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "behaviour", "fold"],
    ]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(["Ind_ID", "new_burst", "behaviour", "fold"], axis=1)
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        # The N_INNER_FOLDS folds following test_fold are the inner
        # validation folds; each inner round trains on the remaining 3
        # folds (excluding test_fold and that round's validation fold).
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_xgb_fold(
            cv_folds,
            X_full[non_test],
            y_enc[non_test],
            X_full[te],
            y_enc[te],
            class_names,
            burst,
            correction_type,
            test_fold,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        # Kept for the post-hoc "Inspect best study per dataset" cell.
        out["studies"][test_fold] = study

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "max_depth",
        "learning_rate",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    # Default-control metrics: no val_f1_macro/default_val_f1_macro/
    # search_time columns, since no search is run for them.
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "max_depth",
        "learning_rate",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }


## Run XGBoost tuning across datasets

In [ ]:
all_results_xgb = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_xgb(pq_file_path, fold_lookup_all, n_trials=N_TRIALS)
    all_results_xgb.append(result)

multiclass_metrics_xgb = pd.concat(
    [r["multiclass_metrics"] for r in all_results_xgb], ignore_index=True
)
multiclass_conf_xgb = pd.concat(
    [r["multiclass_conf"] for r in all_results_xgb], ignore_index=True
)
default_metrics_xgb = pd.concat(
    [r["default_metrics"] for r in all_results_xgb], ignore_index=True
)
default_conf_xgb = pd.concat(
    [r["default_conf"] for r in all_results_xgb], ignore_index=True
)

# Kept in memory (not saved) for the "Inspect best study per dataset
# (XGBoost)" cell, same structure as all_studies_rf.
all_studies_xgb = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_xgb)
}

## SVC Optuna search + final evaluation, for one outer test fold

In [ ]:
def eval_svc_cv_folds(cv_folds, C, gamma, use_class_weight=True):
    """Fit+score one SVC hyperparameter setting across all of cv_folds
    (the N_INNER_FOLDS inner train/val splits), returning the mean
    f1_macro.

    Mirrors eval_rf_cv_folds; shared by make_svc_objective's Optuna
    trials and the SVC_DEFAULT_PARAMS control in tune_and_evaluate_svc_fold.

    Uses probability=False: Platt scaling (probability=True) runs an
    internal 5-fold CV on every fit, and the search only needs .predict().
    probability=True is used only in retrain_and_evaluate_svc, where
    roc_auc needs predict_proba.
    """
    cw = "balanced" if use_class_weight else None
    scores = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for X_train, y_train, X_val, y_val in cv_folds:
            # RBF kernel is distance-based, so features are standardised
            # within each training split (the scaler is fitted on X_train only)
            model = make_pipeline(
                StandardScaler(),
                SVC(
                    kernel="rbf",
                    cache_size=1000,
                    probability=False,
                    C=C,
                    gamma=gamma,
                    class_weight=cw,
                    random_state=RANDOM_STATE,
                ),
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))
    return float(np.mean(scores))


def make_svc_objective(cv_folds, use_class_weight=True):
    """Build an Optuna objective for SVC that returns the mean f1_macro
    across the N_INNER_FOLDS inner validation folds. Searches C
    (regularization strength, 0.1-100, log scale) and gamma (RBF kernel
    bandwidth, 1e-4-1, log scale, on standardised features).
    """

    def objective(trial):
        C = trial.suggest_float("C", 0.1, 100, log=True)
        gamma = trial.suggest_float("gamma", 1e-4, 1, log=True)
        return eval_svc_cv_folds(cv_folds, C, gamma, use_class_weight=use_class_weight)

    return objective


def retrain_and_evaluate_svc(
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    params,
    class_names,
    use_class_weight=True,
):
    """Fit one SVC hyperparameter setting on X_train_val/y_train_val and
    evaluate once on X_test/y_test, returning (metrics_row,
    confusion_rows) -- confusion_rows is a list of dicts without a
    "model" key (the caller tags that).

    Mirrors retrain_and_evaluate_rf. Shared by tune_and_evaluate_svc_fold's
    two final retrains: the tuned winner and the SVC_DEFAULT_PARAMS
    control. Uses probability=True, since roc_auc needs predict_proba.
    """
    cw = "balanced" if use_class_weight else None
    n_classes = len(class_names)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        # Same StandardScaler pipeline as eval_svc_cv_folds
        model = make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                cache_size=1000,
                probability=True,
                C=params["C"],
                gamma=params["gamma"],
                class_weight=cw,
                random_state=RANDOM_STATE,
            ),
        )
        model.fit(X_train_val, y_train_val)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        try:
            roc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        metrics_row = {
            "C": params["C"],
            "gamma": params["gamma"],
            "roc_auc": roc,
            "accuracy": model.score(X_test, y_test),
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "precision": precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "precision_macro": precision_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "time": time.time() - t0,
        }

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
        confusion_rows = [
            {
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(n_classes)
            for j in range(n_classes)
        ]

    return metrics_row, confusion_rows


def tune_and_evaluate_svc_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for SVC. Same protocol as
    tune_and_evaluate_rf_fold. SVC_DEFAULT_PARAMS (gamma="scale") cannot
    be expressed in the search space, so, as for RF, the default control
    is evaluated directly via eval_svc_cv_folds and
    retrain_and_evaluate_svc.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). Same shape as
            tune_and_evaluate_rf_fold's return, with model="SVC" /
            "SVC_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )

    search_start = time.time()
    study.optimize(
        make_svc_objective(cv_folds, use_class_weight=use_class_weight),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    default_val_f1_macro = eval_svc_cv_folds(
        cv_folds, **SVC_DEFAULT_PARAMS, use_class_weight=use_class_weight
    )

    tuned_row, tuned_conf_rows = retrain_and_evaluate_svc(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        best_params,
        class_names,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "SVC",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "SVC", **row} for row in tuned_conf_rows]

    # Default-hyperparameter control, retrained on the same split as the
    # tuned retrain above.
    default_row, default_conf_rows = retrain_and_evaluate_svc(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        SVC_DEFAULT_PARAMS,
        class_names,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "SVC_default"
    default_confusion_data = [
        {"model": "SVC_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## SVC tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_svc(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for SVC
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Same structure as process_parquet_file_rf, with
    tune_and_evaluate_svc_fold as the per-fold trainer and C/gamma as the
    hyperparameter columns.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup_all.loc[
        fold_lookup_all["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "behaviour", "fold"],
    ]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(["Ind_ID", "new_burst", "behaviour", "fold"], axis=1)
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        # The N_INNER_FOLDS folds following test_fold are the inner
        # validation folds; each inner round trains on the remaining 3
        # folds (excluding test_fold and that round's validation fold).
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_svc_fold(
            cv_folds,
            X_full[non_test],
            y_enc[non_test],
            X_full[te],
            y_enc[te],
            class_names,
            burst,
            correction_type,
            test_fold,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        # Kept for the post-hoc "Inspect best study per dataset" cell.
        out["studies"][test_fold] = study

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "C",
        "gamma",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    # Default-control metrics: no val_f1_macro/default_val_f1_macro/
    # search_time columns, since no search is run for them.
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "C",
        "gamma",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }


## Run SVC tuning across datasets

In [ ]:
all_results_svc = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_svc(pq_file_path, fold_lookup_all, n_trials=N_TRIALS)
    all_results_svc.append(result)

multiclass_metrics_svc = pd.concat(
    [r["multiclass_metrics"] for r in all_results_svc], ignore_index=True
)
multiclass_conf_svc = pd.concat(
    [r["multiclass_conf"] for r in all_results_svc], ignore_index=True
)
default_metrics_svc = pd.concat(
    [r["default_metrics"] for r in all_results_svc], ignore_index=True
)
default_conf_svc = pd.concat(
    [r["default_conf"] for r in all_results_svc], ignore_index=True
)

# Kept in memory (not saved) for the "Inspect best study per dataset
# (SVC)" cell, same structure as all_studies_rf.
all_studies_svc = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_svc)
}

## Pool results and compare tuned vs default

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-fold metrics when classes are unevenly represented across folds.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())


    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def pooled_summary_per_dataset(conf_df, model_name):
    """Pooled metrics computed separately per (burst, correction_type), so
    datasets are never mixed into a single pool."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(grp, model_name)
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


def _run_df(name):
    """Return the in-memory DataFrame `name` if that model's cells were
    (re-)run this session, else an empty DataFrame.

    Each model's cells can be run on their own, so not every model's
    frames necessarily exist in this session. Models not run are skipped,
    and their rows already on disk are left untouched by upsert_csv.
    """
    return globals().get(name, pd.DataFrame())


# Combine this session's RF / XGB / SVC results (whichever were run) into
# single frames, before merging with the CSVs on disk in the save cell.
multiclass_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_metrics_rf"),
            _run_df("multiclass_metrics_xgb"),
            _run_df("multiclass_metrics_svc"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
multiclass_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_conf_rf"),
            _run_df("multiclass_conf_xgb"),
            _run_df("multiclass_conf_svc"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_metrics_rf"),
            _run_df("default_metrics_xgb"),
            _run_df("default_metrics_svc"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_conf_rf"),
            _run_df("default_conf_xgb"),
            _run_df("default_conf_svc"),
        )
        if not df.empty
    ],
    ignore_index=True,
)

# Only the models present in this session (see _run_df).
MODEL_NAMES = sorted(multiclass_conf_run["model"].unique())
print(f"Models pooled this run: {MODEL_NAMES}")

pooled_multiclass_summary = pd.concat(
    [pooled_summary_per_dataset(multiclass_conf_run, m) for m in MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled tuned multiclass metrics per dataset:")
print(pooled_multiclass_summary)

# Per-fold final test performance, with the validation-side comparison
# of tuned (val_f1_macro) vs default (default_val_f1_macro).
# Model-specific hyperparameter columns are omitted from this print but
# kept in multiclass_metrics_run and the saved CSV.
per_fold_cols = [
    "burst",
    "correction_type",
    "test_fold",
    "val_folds",
    "n_test_samples",
    "model",
    "val_f1_macro",
    "default_val_f1_macro",
    "accuracy",
    "balanced_acc",
    "f1_macro",
    "search_time",
    "time",
]
per_fold_multiclass = multiclass_metrics_run[per_fold_cols].sort_values(
    ["burst", "correction_type", "model", "test_fold"]
)
print(
    "\nPer-fold tuned multiclass results (final test metrics; see "
    "multiclass_metrics_run for the full per-model hyperparameter columns):"
)
print(per_fold_multiclass)

# Number of outer folds where the search beat the default on the inner
# validation folds, and the mean gain.
beats_default = (
    per_fold_multiclass["val_f1_macro"] > per_fold_multiclass["default_val_f1_macro"]
)
print(
    f"\nSearch beat the default on validation in {beats_default.sum()}/{len(beats_default)} "
    f"outer folds across {MODEL_NAMES} (mean val_f1_macro - default_val_f1_macro = "
    f"{(per_fold_multiclass['val_f1_macro'] - per_fold_multiclass['default_val_f1_macro']).mean():.4f})"
)

# Default control on the test folds: pooled tuned vs pooled default
# performance, both retrained and evaluated on the same splits.
DEFAULT_MODEL_NAMES = sorted(default_conf_run["model"].unique())
pooled_default_summary = pd.concat(
    [pooled_summary_per_dataset(default_conf_run, m) for m in DEFAULT_MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled DEFAULT (test-fold) metrics per dataset:")
print(pooled_default_summary)

# Pair each tuned model's pooled summary with its "<model>_default"
# counterpart (merging on model_base as well as burst/correction_type).
test_gain = pooled_multiclass_summary.assign(
    model_base=pooled_multiclass_summary["model"]
).merge(
    pooled_default_summary.assign(
        model_base=pooled_default_summary["model"].str.replace(
            "_default", "", regex=False
        )
    ),
    on=["burst", "correction_type", "model_base"],
    suffixes=("_tuned", "_default"),
)[["burst", "correction_type", "model_base", "f1_macro_tuned", "f1_macro_default"]]
test_gain["f1_macro_gain"] = test_gain["f1_macro_tuned"] - test_gain["f1_macro_default"]
print("\nTest-set gain (tuned - default, pooled f1_macro), per model:")
print(test_gain)

## Inspect best study per dataset

Post-hoc, after all outer folds are complete. For a chosen dataset, picks
the outer fold with the highest validation `f1_macro` and shows that
fold's Optuna study summary and plots.

Change `INSPECT_DATASET` to any tuple from `DATASETS` to look at a
different dataset.


In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_RF = DATASETS[0]

dataset_rows_rf = multiclass_metrics_rf[
    (multiclass_metrics_rf["burst"].str.lower() == INSPECT_DATASET_RF[0])
    & (multiclass_metrics_rf["correction_type"] == INSPECT_DATASET_RF[1])
]
best_row_rf = dataset_rows_rf.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_rf = best_row_rf["test_fold"]
study_rf = all_studies_rf[INSPECT_DATASET_RF][best_test_fold_rf]

complete_trials_rf = [
    t for t in study_rf.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_RF} "
    f"(by validation f1_macro across its {len(dataset_rows_rf)} outer folds): "
    f"test_fold={best_test_fold_rf}"
)
print(f"  # trials (completed) : {len(complete_trials_rf)}/{len(study_rf.trials)}")
print(f"  Best value            : {study_rf.best_value:.4f}")
print(f"  Best params           : {study_rf.best_params}")
print(f"  Default params        : {RF_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_rf['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_rf.best_value - best_row_rf['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_rf).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Inspect best study per dataset (XGBoost)

Same as the RF cell above.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_XGB = DATASETS[0]

dataset_rows_xgb = multiclass_metrics_xgb[
    (multiclass_metrics_xgb["burst"].str.lower() == INSPECT_DATASET_XGB[0])
    & (multiclass_metrics_xgb["correction_type"] == INSPECT_DATASET_XGB[1])
]
best_row_xgb = dataset_rows_xgb.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_xgb = best_row_xgb["test_fold"]
study_xgb = all_studies_xgb[INSPECT_DATASET_XGB][best_test_fold_xgb]

complete_trials_xgb = [
    t for t in study_xgb.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_XGB} "
    f"(by validation f1_macro across its {len(dataset_rows_xgb)} outer folds): "
    f"test_fold={best_test_fold_xgb}"
)
print(f"  # trials (completed) : {len(complete_trials_xgb)}/{len(study_xgb.trials)}")
print(f"  Best value            : {study_xgb.best_value:.4f}")
print(f"  Best params           : {study_xgb.best_params}")
print(f"  Default params        : {XGB_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_xgb['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_xgb.best_value - best_row_xgb['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_xgb).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Inspect best study per dataset (SVC)

Same as the RF cell above.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_SVC = DATASETS[0]

dataset_rows_svc = multiclass_metrics_svc[
    (multiclass_metrics_svc["burst"].str.lower() == INSPECT_DATASET_SVC[0])
    & (multiclass_metrics_svc["correction_type"] == INSPECT_DATASET_SVC[1])
]
best_row_svc = dataset_rows_svc.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_svc = best_row_svc["test_fold"]
study_svc = all_studies_svc[INSPECT_DATASET_SVC][best_test_fold_svc]

complete_trials_svc = [
    t for t in study_svc.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_SVC} "
    f"(by validation f1_macro across its {len(dataset_rows_svc)} outer folds): "
    f"test_fold={best_test_fold_svc}"
)
print(f"  # trials (completed) : {len(complete_trials_svc)}/{len(study_svc.trials)}")
print(f"  Best value            : {study_svc.best_value:.4f}")
print(f"  Best params           : {study_svc.best_params}")
print(f"  Default params        : {SVC_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_svc['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_svc.best_value - best_row_svc['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_svc).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Helper: merge new results into existing output CSVs

In [ ]:
def upsert_csv(new_df, path, key_cols):
    """Merge new_df's rows into the CSV at path (creating it if it
    doesn't exist yet), keeping new_df's rows wherever key_cols collide
    with what's already on disk.

    This lets each model's tuning cells be run independently without
    dropping the other models' rows from the shared output files. Called
    from the save cell, once per output file.
    """
    if os.path.exists(path):
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=key_cols, keep="last")
        # Drop columns that are empty after the merge, e.g. hyperparameter
        # columns of a replaced search space (RF/XGB n_estimators).
        combined = combined.dropna(axis=1, how="all")
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return combined

## Save outputs

In [ ]:
# Merges this session's results (built in the pooling cell) into the
# output CSVs via upsert_csv, so rows for models not run this session are
# kept. Only the behaviour target is saved.
METRIC_KEY = ["burst", "correction_type", "target", "test_fold", "model"]
CONF_KEY = [
    "burst",
    "correction_type",
    "target",
    "test_fold",
    "model",
    "actual_label",
    "predicted_label",
]

upsert_csv(
    multiclass_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_ml_tuned_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    multiclass_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_ml_tuned_confusion.csv",
    CONF_KEY,
)

# Default-hyperparameter control (test-fold retrain), saved to separate
# files so the tuned metrics/confusion files read by
# 12b_publication_plots_fig3.R contain only tuned rows.
upsert_csv(
    default_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_ml_tuned_default_control_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    default_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_ml_tuned_default_control_confusion.csv",
    CONF_KEY,
)